# 02 - Run Baselines
Train teacher and classic students, check results.

**Saved outputs:** this notebook writes baseline fold metrics to `../outputs/tables/student_classic_folds.csv` and `../outputs/tables/student_classic_kd_folds.csv`, and also writes teacher reports to `../outputs/tables/teacher_summary.json` and `../outputs/tables/teacher_entropy.json`. Notebook 05 rebuilds `summary.json` from the fold CSV files.


In [ ]:
import sys; sys.path.insert(0, "..")
from src.analysis.teacher_stats import save_teacher_reports
from src.utils.seed import set_seed
from src.trainers.train_teacher import train_teacher_cv
from src.trainers.train_student import run_student_cv

set_seed(42)
CSV = "../data/raw/wdbc.csv"
teacher = train_teacher_cv(CSV, pca_dim=4)
teacher_reports = save_teacher_reports(teacher)
print("Teacher entropy (train):", teacher_reports["entropy"]["train"])
print("Teacher entropy (val):", teacher_reports["entropy"]["val"])
run_student_cv(CSV, model_type="classic", use_kd=False, exp_name="student_classic")
run_student_cv(CSV, teacher_fold_outputs=teacher, model_type="classic", use_kd=True, exp_name="student_classic_kd")


In [ ]:
import json, numpy as np

# ── Enhanced teacher entropy analysis ──
# Uses in-memory teacher fold outputs (from train_teacher_cv above).
# Produces min / max in addition to the mean / median / std / high-conf fraction
# that save_teacher_reports already writes.

tr_logits = np.concatenate(
    [fold["tr_logits"].detach().cpu().numpy().reshape(-1) for fold in teacher]
)
va_logits = np.concatenate(
    [fold["va_logits"].detach().cpu().numpy().reshape(-1) for fold in teacher]
)


def _binary_entropy(logits):
    probs = 1.0 / (1.0 + np.exp(-logits))
    probs = np.clip(probs, 1e-6, 1 - 1e-6)
    return -probs * np.log2(probs) - (1 - probs) * np.log2(1 - probs), probs


enhanced = {}
for split_name, logits in [("train", tr_logits), ("val", va_logits)]:
    ent, probs = _binary_entropy(logits)
    stats = {
        "split": split_name,
        "num_samples": int(len(ent)),
        "mean_entropy_bits": float(np.mean(ent)),
        "median_entropy_bits": float(np.median(ent)),
        "std_entropy_bits": float(np.std(ent)),
        "min_entropy_bits": float(np.min(ent)),
        "max_entropy_bits": float(np.max(ent)),
        "frac_abs_p_minus_0_5_gt_0_45": float(np.mean(np.abs(probs - 0.5) > 0.45)),
    }
    enhanced[split_name] = stats
    print(f"\n{split_name}: {json.dumps(stats, indent=2)}")

# Overwrite teacher_entropy.json with the enhanced version (adds min / max)
with open("../outputs/tables/teacher_entropy.json", "w") as f:
    json.dump(enhanced, f, indent=2)
print("\n✓ Saved enhanced teacher_entropy.json")